# Case Study 01 — Model Training & Validation

Builds the credit scorecard and validates it with industry-standard metrics:
KS, Gini, AUC-ROC, CAP curve, PSI (population stability).

**Prerequisite:** Run `02_feature_engineering.ipynb` first.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

REPO_ROOT = Path().resolve().parents[1]
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from utils.metrics import ks_statistic, gini_coefficient, psi, cap_ratio
from utils.plotting import plot_cap_curve, plot_roc_curve, plot_score_distribution
from src.scorecard import Scorecard

DATA_DIR = Path('../data')
REPORTS_DIR = Path('../reports')
TARGET = 'default'

df = pd.read_parquet(DATA_DIR / 'processed_woe.parquet')
X = df.drop(columns=[TARGET])
y = df[TARGET]
print(f'Dataset: {df.shape} | Default rate: {y.mean():.2%}')

## 1. Train/Test Split (OOT simulation: last 20%)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(f'Train: {X_train.shape} | Default: {y_train.mean():.2%}')
print(f'Test : {X_test.shape}  | Default: {y_test.mean():.2%}')

## 2. Logistic Regression — Regulatory Standard Model

In [ ]:
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('lr', LogisticRegression(C=1.0, max_iter=500, random_state=42, class_weight='balanced'))
])
pipe.fit(X_train, y_train)

# Cross-validation on training set
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_auc = cross_val_score(pipe, X_train, y_train, cv=cv, scoring='roc_auc')
print(f'CV AUC: {cv_auc.mean():.4f} ± {cv_auc.std():.4f}')

## 3. Scorecard Scaling (PDO Methodology)

In [ ]:
lr_model = pipe.named_steps['lr']
scaler = pipe.named_steps['scaler']

# Create scorecard with PDO=20, base_score=600, base_odds=50
scorecard = Scorecard(lr_model, pdo=20, base_score=600, base_odds=50)

X_test_scaled = scaler.transform(X_test)
X_train_scaled = scaler.transform(X_train)

# Predict on test set
y_prob_test = scorecard.predict_proba(pd.DataFrame(X_test_scaled, columns=X_test.columns))
y_score_test = scorecard.predict_score(pd.DataFrame(X_test_scaled, columns=X_test.columns))

print(f'Score range: {y_score_test.min():.0f} — {y_score_test.max():.0f}')
print(f'Score mean : {y_score_test.mean():.0f}')

## 4. Validation Metrics

In [ ]:
y_true_arr = y_test.values

ks   = ks_statistic(y_true_arr, y_prob_test)
gini = gini_coefficient(y_true_arr, y_prob_test)
cap  = cap_ratio(y_true_arr, y_prob_test)

# PSI: train vs test score distributions
y_prob_train = scorecard.predict_proba(pd.DataFrame(X_train_scaled, columns=X_train.columns))
psi_val = psi(y_prob_train, y_prob_test)

results = {
    'KS Statistic'    : (ks,      '> 0.30', 'PASS' if ks > 0.30 else 'FAIL'),
    'Gini Coefficient': (gini,    '> 0.40', 'PASS' if gini > 0.40 else 'FAIL'),
    'AUC-ROC'         : (gini/2 + 0.5, '> 0.70', 'PASS' if (gini/2+0.5) > 0.70 else 'FAIL'),
    'CAP Ratio'       : (cap,     '> 0.60', 'PASS' if cap > 0.60 else 'FAIL'),
    'PSI'             : (psi_val, '< 0.10', 'PASS' if psi_val < 0.10 else 'MONITOR'),
}

print('=== Model Validation Report ===')
print(f'{"Metric":<22} {"Value":>8}  {"Threshold":>12}  {"Status"}')
print('-' * 55)
for metric, (val, thr, status) in results.items():
    icon = '✓' if 'PASS' in status else '✗'
    print(f'{metric:<22} {val:>8.4f}  {thr:>12}  {icon} {status}')

## 5. Validation Charts

In [ ]:
fig1 = plot_cap_curve(y_true_arr, y_prob_test, title='CAP Curve — Credit Scoring Model',
                      save_path=str(REPORTS_DIR / '08_cap_curve.png'))
plt.show()

fig2 = plot_roc_curve(y_true_arr, y_prob_test, title='ROC Curve — Credit Scoring Model',
                      save_path=str(REPORTS_DIR / '09_roc_curve.png'))
plt.show()

fig3 = plot_score_distribution(y_true_arr, y_prob_test,
                               title='Score Distribution — Good vs Bad Accounts',
                               save_path=str(REPORTS_DIR / '10_score_distribution.png'))
plt.show()
print('All validation charts saved to reports/')